# Explorative Datenanalyse (EDA) der aufbereiteten Metadaten

author: "Zhanna Davtyan"

***Hinweis:*** führen Sie dieses Notebook aus, nachdem Sie das `02 Datenbereinigung und Vorbereitung` Notebook ausgeführt haben, um die erforderlichen Metadaten zu generieren.

Die explorative Datenanalyse umfasste umfangreiche Visualisierungen, um Muster und Beziehungen zwischen den Features und der Zielvariable zu identifizieren.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import os
from pathlib import Path

In [ ]:
base_dir = Path.cwd()
if base_dir.name == "notebooks":
    base_dir = base_dir.parent
    os.chdir(base_dir)

In [ ]:
df_gefiltert = pd.read_pickle('ergebnisse/verarbeitete_daten.pkl')

### Bereitstellung des DataFrames für die Modellierung

In [ ]:
# Auswahl der Features für das Modell
feature_columns = [
    'follower_count', 'is_verified', 'account_category', 'is_business_account',
    'is_professional_account', 'comment_count', 'post_type',
    'caption_length', 'hashtag_count', 'mention_count', 'tagged_user_count', 
    'media_count_in_post', 'has_location',
    'hour_of_day', 'day_of_week', 'month'
]
existing_feature_columns = [col for col in feature_columns if col in df_gefiltert.columns]
      

X = df_gefiltert[existing_feature_columns].copy()
y = df_gefiltert['likes_log'].copy()

print("\nAusgewählte Features für X:")
X.info()

# Fehlende Werte in 'account_category' mit 'Unbekannt' füllen 
if 'account_category' in X.columns:
    X['account_category'] = X['account_category'].fillna('Unbekannt')
    
print("\nFehlende Werte in X nach Imputation für 'account_category':")
print(X.isnull().sum())

X

### Verteilungsanalyse
Die Verteilung der Like-Anzahl zeigte eine stark rechtsschiefe Verteilung, was die Log-Transformation rechtfertigte:

- Die Originalverteilung der Likes wies extreme Ausreißer auf
- Die log-transformierte Variable (likes_log) zeigte eine wesentlich gleichmäßigere Verteilung


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(df_gefiltert['likes_log'], kde=True, bins=50, ax=axes[0])
axes[0].set_title('Verteilung der log-transformierten Likes (log(1+Likes))')
axes[0].set_xlabel('Log(1 + Anzahl Likes)')
axes[0].set_ylabel('Häufigkeit')

sns.histplot(df_gefiltert['likes'], kde=True, bins=50, ax=axes[1])
axes[1].set_title('Verteilung der Likes')
axes[1].set_xlabel('Anzahl Likes')
axes[1].set_ylabel('Häufigkeit')

plt.tight_layout()
plt.show()



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(x=df_gefiltert['likes_log'], ax=axes[0])
axes[0].set_title('Boxplot der log-transformierten Likes')
axes[0].set_xlabel('Log(1 + Anzahl Likes)')

sns.boxplot(x=df_gefiltert['likes'], ax=axes[1])
axes[1].set_title('Boxplot der Likes')
axes[1].set_xlabel('Anzahl Likes')

plt.tight_layout()
plt.show()

### Korrelationen numerischer Merkmale

Um eine aussagekräftigere Analyse zu erhalten, sollen die numerischen Features standardisiert (z-Transformation) werden, damit sie alle den gleichen Maßstab haben. 


In [ ]:
num_merkmale_corr = [col for col in X.select_dtypes(include=np.number).columns if col in X.columns]

temp_df = X[num_merkmale_corr].copy()
temp_df['likes_log'] = y 

df_std = temp_df.copy()

# StandardScaler 
scaler = StandardScaler()
numeric_features = [col for col in df_std.columns if col != 'likes_log']
df_std[numeric_features] = scaler.fit_transform(df_std[numeric_features])


correlation_matrix = df_std.corr()
plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f",  cmap="RdBu", center = 0)
plt.title('Korrelationsmatrix der numerischen Features und log(Likes)')
plt.show()

### Interpretation von Korrelationsmatrix:

Um eine Überblick über die einzelne prädikatoren zu erhalten, wird ein pairplot erstellt, der die Korrelationen zwischen den numerischen Merkmalen visualisiert. 
Die Likes-Log-Transformation wird als Zielvariable in Abhängigkeit von `follower_count`, `comment_count`, 
`caption_length`, `hashtag_count`, `mention_count`, `media_count_in_post`, `hour_of_day` verwendet.

1. Stärkste positive Korrelation:

    - follower_count hat eine Korrelation von +0.37 mit likes_log. Das ist die stärkste positive Korrelation in dieser Matrix mit likes_log. Ein Wert von 0.37 deutet auf einen positiven, aber eher moderaten Zusammenhang hin. Das bedeutet, tendenziell haben Profile mit mehr Followern auch Posts mit mehr Likes (bzw. einem höheren Log-Wert der Likes).

2. Stärkste negative Korrelation:

    - hashtag_count hat eine Korrelation von -0.40 mit likes_log. Dies ist die stärkste negative Korrelation. Ein Wert von -0.40 deutet auf einen moderaten negativen Zusammenhang hin. Das könnte bedeuten, dass Posts mit einer höheren Anzahl von Hashtags tendenziell etwas weniger Likes bekommen. Zu viele Hashtags wirken möglicherweise spammy oder unnatürlich.


In [ ]:
# Korrelationen sortieren und Top-3 Features auswählen
corrs_std = df_std.corr()['likes_log'].abs().sort_values(ascending=False)
top_features_std = [col for col in corrs_std.index if col != 'likes_log'][:3]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feature in enumerate(top_features_std):
    # Regression mit standardisierten Werten 
    sns.regplot(
        x=feature, y='likes_log', data=df_std,
        scatter_kws={'s': 20, 'alpha': 0.5, 'edgecolor': 'k' },
        ax=axes[i]
    )
    axes[i].set_title(f'Log(Likes) vs. {feature} (standardisiert)')
    axes[i].set_xlabel(f'{feature} (standardisiert)')
    axes[i].set_ylabel('Log(1 + Anzahl Likes)')
plt.tight_layout()
plt.show()

### Kategoriale Merkmale 

Für kategoriale Merkmale wie is_verified, account_category und is_business_account wurden Boxplots erstellt, um Unterschiede in der Like-Verteilung zu visualisieren. 

In [ ]:
kategoriale_merkmale = [col for col in X.select_dtypes(include=['object', 'bool', 'category']).columns]


for kategorial in kategoriale_merkmale:

    temporaere_plot_df = pd.concat([X[[kategorial]].copy(), y.copy()], axis=1)
    
    plt.figure(figsize=(14, 8))
    
    # Spezialbehandlung für 'account_category'
    if kategorial == 'account_category':
        
        top_n = temporaere_plot_df[kategorial].value_counts().nlargest(10).index # Top 10 Kategorien
        # Alle Kategorien, die nicht zu den Top 10 gehören - werden als 'Andere' zusammengefasst
        temporaere_plot_df[kategorial] = temporaere_plot_df[kategorial].astype(str).apply(lambda x: x if x in top_n else 'Andere')
        order = temporaere_plot_df.groupby(kategorial)['likes_log'].median().sort_values(ascending=False).index
        sns.boxplot(x=kategorial, y='likes_log', data=temporaere_plot_df, order=order, hue=kategorial, palette='viridis', legend=False)
    else:
        # ALLGEMEINER FALL: Für ALLE ANDEREN Merkmale in der Liste "kategoriale_merkmale"

        order = temporaere_plot_df.groupby(kategorial)['likes_log'].median().sort_values(ascending=False).index
        sns.boxplot(x=kategorial, y='likes_log', data=temporaere_plot_df, order=order, hue=kategorial, palette='viridis', legend=False)

    plt.title(f'Log(Likes) nach {kategorial}', fontsize=16)
    plt.xlabel(kategorial, fontsize=14)
    plt.ylabel('Log(1 + Anzahl Likes)', fontsize=14)
    if temporaere_plot_df[kategorial].nunique() > 5 : 
        plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.tight_layout()
    plt.show()


### Interpretation der Ergebnisse:

Besonders auffällige Ergebnisse aus den Boxplots:

- Verifizierte Accounts erhielten signifikant mehr Likes
- Business-Accounts zeigten eine andere Like-Verteilung als normale Accounts
- Account-Kategorien wiesen deutliche Unterschiede in der durchschnittlichen Like-Anzahl auf
